# Buổi 5 · Tình huống 03 — Cảnh báo giao hàng trễ (Adaline)

Notebook này dùng lại bối cảnh trong [`bai-tap/buoi4-tinh-huong/03_giao_hang_tre.py`](../../bai-tap/buoi4-tinh-huong/03_giao_hang_tre.py), nhưng thay Perceptron (0/1) bằng **Adaline** học quy tắc Widrow–Hoff (LMS) với **target liên tục**.

Không có dữ liệu hoặc lời giải dựng sẵn trong notebook này. Các code cell chỉ chứa comment định hướng; bạn tự thu thập dữ liệu, xử lý, huấn luyện, đánh giá và triển khai thử.

## Bối cảnh

Một cửa hàng muốn biết đơn hàng nào có nguy cơ giao trễ để chủ động thông báo.

Feature có thể cân nhắc: khoảng cách, số đơn tài xế đang giữ, trời mưa, giờ cao điểm, thời gian chuẩn bị (mã hoá số, chuẩn hoá).

## Vì sao dùng Adaline thay Perceptron?

Thay vì chỉ 0/1 (đúng giờ / trễ), hãy coi target là **số phút trễ dự kiến** hoặc **điểm nguy cơ trễ** trong [0, 1]. Nhiều đơn sẽ có nguy cơ trung gian, không chỉ hai cực.

Nhắc lại công thức Adaline / Widrow–Hoff:

$$
\text{net} = \mathbf{w}^\top \mathbf{x} + b, \qquad e = t - \text{net}
$$

$$
\mathbf{w} \leftarrow \mathbf{w} + \eta e \mathbf{x}, \qquad b \leftarrow b + \eta e
$$

Với target là số phút trễ dự kiến, sai số MSE cho biết mức độ lệch cụ thể, chi tiết hơn so với chỉ biết đúng/sai như Perceptron.

## Nhiệm vụ suy nghĩ

1. Quyết định cách biến trời mưa và giờ cao điểm thành số (0/1 hoặc mức độ).
2. Tạo dữ liệu có cả đơn dễ đoán và đơn nằm gần ranh giới quyết định, với target liên tục.
3. Chia train/test trước khi huấn luyện Adaline.
4. Theo dõi MSE trên train và test qua các epoch; so sánh với việc chỉ đo accuracy.
5. Chọn ngưỡng cảnh báo và cân nhắc coi trọng phát hiện đơn có nguy cơ trễ cao (recall) hay tránh cảnh báo nhầm (precision).

Không có code mẫu hoặc lời giải trong notebook này. Bắt đầu phần triển khai ở các cell dưới.

## 1. Thu thập / tạo dữ liệu

Tự tạo hoặc thu thập một bảng dữ liệu nhỏ. Ghi rõ ý nghĩa, đơn vị của từng feature và cách bạn gán target liên tục (không chỉ 0/1).

In [ ]:
import csv
import numpy as np

# 1. Khai báo danh sách các feature theo thứ tự
FEATURE_NAMES = [
    'khoang_cach_km', 
    'so_don_tai_xe', 
    'troi_mua', 
    'gio_cao_diem', 
    'thoi_gian_chuan_bi_phut'
]

X_list = []
t_list = []

# 2. Đọc dữ liệu từ file CSV bằng module csv thuần
with open('du_lieu_giao_muon_50000.csv', mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Ép kiểu dữ liệu về float cho 5 đặc trưng
        X_list.append([
            float(row['khoang_cach_km']),
            float(row['so_don_tai_xe']),
            float(row['troi_mua']),
            float(row['gio_cao_diem']),
            float(row['thoi_gian_chuan_bi_phut'])
        ])
        # Nạp target t dạng liên tục trong [0, 1]
        t_list.append(float(row['kha_nang_giao_muon']))

# 3. Chuyển đổi thành mảng NumPy
X_raw = np.array(X_list, dtype=float)  # Shape: (50000, 5)
t_raw = np.array(t_list, dtype=float)  # Shape: (50000,)

print(f"-> Nạp dữ liệu thành công!")
print(f"Kích thước ma trận đặc trưng X_raw: {X_raw.shape}")
print(f"Kích thước vector nhãn target t_raw : {t_raw.shape}")

## 2. Chuẩn hoá feature

Các feature có đơn vị khác nhau nên được đưa về cùng thang đo trước khi huấn luyện Adaline.

In [ ]:
X_mean = X_train_raw.mean(axis=0)  # Vector trung bình của 5 đặc trưng trên tập train
X_std = X_train_raw.std(axis=0)    # Vector độ lệch chuẩn của 5 đặc trưng trên tập train

# Áp dụng tham số chuẩn hoá nhất quán cho cả tập Train và tập Test
X_train = (X_train_raw - X_mean) / X_std
X_test = (X_test_raw - X_mean) / X_std

print("-> Chuẩn hoá Z-score hoàn tất!")
print(f"Lưu tham số X_mean: {np.round(X_mean, 2)}")
print(f"Lưu tham số X_std : {np.round(X_std, 2)}")

## 3. Tách train/test

Tách dữ liệu trước khi huấn luyện để có thể đánh giá khách quan.

In [ ]:
np.random.seed(42)  # Cố định ngẫu nhiên để kết quả có thể tái lập

# Tạo mảng chỉ số ngẫu nhiên từ 0 đến len(X_raw) - 1
indices = np.arange(len(X_raw))
np.random.shuffle(indices)

# Tính số lượng mẫu cho tập Train (80%)
train_size = int(0.8 * len(X_raw))
train_idx = indices[:train_size]  # 40.000 mẫu đầu tiên làm Train
test_idx = indices[train_size:]   # 10.000 mẫu còn lại làm Test

# Tách ma trận đặc trưng thô và nhãn target
X_train_raw, X_test_raw = X_raw[train_idx], X_raw[test_idx]
t_train, t_test = t_raw[train_idx], t_raw[test_idx]

print(f"-> Tách dữ liệu thành công!")
print(f"Kích thước tập Train (X_train_raw): {X_train_raw.shape}, t_train: {t_train.shape}")
print(f"Kích thước tập Test  (X_test_raw) : {X_test_raw.shape}, t_test : {t_test.shape}")

## 4. Cài đặt và huấn luyện Adaline

Viết (hoặc tái sử dụng) một bước cập nhật Widrow–Hoff, rồi huấn luyện qua nhiều epoch và theo dõi MSE.

In [ ]:
np.random.seed(42)                      # Cố định seed -> kết quả xáo trộn lặp lại được
indices = np.arange(len(X_raw))         # Mảng chỉ số 0,1,...,49999
np.random.shuffle(indices)              # Xáo trộn ngẫu nhiên chỉ số (tránh dữ liệu theo thứ tự)

train_size = int(0.8 * len(X_raw))      # 0.8 * 50000 = 40000 mẫu train
train_idx, test_idx = indices[:train_size], indices[train_size:]  # 40000 chỉ số train, 10000 chỉ số test

# Xử lý: lấy các dòng theo chỉ số đã xáo trộn
X_train_raw, X_test_raw = X_raw[train_idx], X_raw[test_idx]  # shape (40000,5) và (10000,5)
t_train, t_test = t_raw[train_idx], t_raw[test_idx]          # shape (40000,) và (10000,)

## 5. Vẽ đường học (MSE theo epoch)

Quan sát MSE có giảm dần và ổn định hay không.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

# Vẽ MSE Train và MSE Test theo Epoch
plt.plot(range(1, len(model.mse_train_) + 1), model.mse_train_, marker='o', label='MSE Train')
plt.plot(range(1, len(model.mse_val_) + 1), model.mse_val_, marker='s', label='MSE Test')

plt.title('Đường học Adaline (MSE theo Epoch) - Cảnh báo giao hàng trễ')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Đánh giá và phân tích

Xem lại các mẫu có sai số dự đoán lớn nhất, đọc dấu của trọng số và đối chiếu với trực giác nghiệp vụ.

In [ ]:
net_test = np.dot(X_test, model.w_) + model.b_
errors_test = t_test - net_test
abs_errors_test = np.abs(errors_test)

# 1. In ra bộ trọng số w và bias b đã học được
print("=============== BẢNG TRỌNG SỐ VÀ BIAS ĐÃ HỌC ===============")
for name, w in zip(FEATURE_NAMES, model.w_):
    print(f"Trọng số {name:25s}: {w:.4f}")
print(f"Bias b                           : {model.b_:.4f}")

# 2. Liệt kê Top 3 mẫu có sai số tuyệt đối |e| lớn nhất trên tập Test
top_3_idx = np.argsort(abs_errors_test)[-3:][::-1]

print("\n=============== TOP 3 MẪU CÓ SAI SỐ |e| LỚN NHẤT =============")
for idx in top_3_idx:
    print(f"Mẫu #{idx}: Feature thô = {X_test_raw[idx]}")
    print(f"  Target thực tế t = {t_test[idx]:.4f} | Dự đoán net = {net_test[idx]:.4f} | Sai số e = {errors_test[idx]:.4f}")

## 7. Chọn ngưỡng và thử nghiệm

Nếu cần một quyết định rời rạc (ví dụ cảnh báo/không cảnh báo) từ điểm liên tục, hãy chọn ngưỡng phù hợp và giải thích lựa chọn.

In [ ]:
# Chọn ngưỡng quyết định (Threshold) = 0.5
# Nếu net >= 0.5 -> Đơn hàng có nguy cơ trễ cao (CẢNH BÁO TRỄ)
# Nếu net < 0.5  -> Đơn hàng nằm trong tầm kiểm soát (BÌNH THƯỜNG)
THRESHOLD = 0.5

def predict_order(record_dict, model_w, model_b, mean_vec, std_vec, threshold=0.5):
    # Lấy đặc trưng thô theo đúng thứ tự
    x_raw = np.array([record_dict[col] for col in FEATURE_NAMES], dtype=float)
    # Chuẩn hoá Z-score bằng Mean và Std của tập Train
    x_scaled = (x_raw - mean_vec) / std_vec
    # Tính điểm net liên tục: net = w^T * x + b
    net = float(np.dot(x_scaled, model_w) + model_b)
    # Đưa ra quyết định rời rạc dựa trên ngưỡng
    decision = "CẢNH BÁO TRỄ" if net >= threshold else "BÌNH THƯỜNG"
    return net, decision

# Định nghĩa 2 mẫu thử nghiệm mới
test_samples = [
    {
        "name": "Đơn hàng A (Điều kiện bất lợi: Khoảng cách xa, mưa, cao điểm)",
        "data": {
            'khoang_cach_km': 12.0,
            'so_don_tai_xe': 5,
            'troi_mua': 1,
            'gio_cao_diem': 1,
            'thoi_gian_chuan_bi_phut': 35.0
        }
    },
    {
        "name": "Đơn hàng B (Điều kiện thuận lợi: Gần, không mưa, ít đơn)",
        "data": {
            'khoang_cach_km': 2.5,
            'so_don_tai_xe': 1,
            'troi_mua': 0,
            'gio_cao_diem': 0,
            'thoi_gian_chuan_bi_phut': 10.0
        }
    }
]

print(f"=== KẾT QUẢ THỬ NGHIỆM VỚI NGƯỠNG THRESHOLD = {THRESHOLD} ===")
for sample in test_samples:
    score, result = predict_order(sample["data"], model.w_, model.b_, X_mean, X_std, THRESHOLD)
    print(f"\n[{sample['name']}]")
    print(f"-> Điểm nguy cơ (net): {score:.4f}")
    print(f"-> Quyết định        : {result}")

## 8. Kết luận

Trả lời ngắn gọn: Adaline với target liên tục có phù hợp hơn Perceptron 0/1 cho tình huống này không? Vì sao? Còn giới hạn gì (ví dụ nếu quan hệ giữa feature và target là phi tuyến, có thể cần MLP)?